# Swap France rows from the France-bundle job into a finished submission
Inputs: challenge data, the finished submission (a notebook output with submission_*/matching_results.tsv or
output/matching_results.tsv), and the France-bundle job output. Writes submission_france/ (+ France decoy filter).

In [ ]:
import os
os.makedirs('/kaggle/working', exist_ok=True)
%cd /kaggle/working

In [ ]:
%%writefile combine_by_country.py
"""Combine two complete submissions by country: rows of the S1 in --countries come from --take, all other rows
from --base. Valid because blocking keys are country-scoped: a pool record is only ever matched to an S1 of
its own country, so rows of different countries never compete for the same record.
Plain Python (no pandas); streams the files.

  python scripts/combine_by_country.py --test-s1 <data>/test/test_source1.tsv \
      --base output_v4 --take output_teammate --countries France --out output_combined

Each of --base / --take is a folder with matching_results.tsv and candidate_pairs.tsv."""
import argparse
import os
from collections import Counter


def s1_countries(path):
    with open(path, encoding="utf-8", newline="") as f:
        head = f.readline().rstrip("\r\n").split("\t")
        i_id, i_ct = head.index("entity_id"), head.index("country")
        return {p[i_id]: p[i_ct] for p in (l.rstrip("\r\n").split("\t") for l in f) if len(p) > max(i_id, i_ct)}


def combine(name, base, take, out, ctry, keep):
    rows = {}
    with open(os.path.join(take, name), encoding="utf-8", newline="") as f:
        header_take = f.readline()
        for line in f:
            s1 = line.split("\t", 1)[0]
            if ctry.get(s1) in keep:
                rows[s1] = line if line.endswith("\n") else line + "\n"
    stats = Counter()
    with open(os.path.join(base, name), encoding="utf-8", newline="") as f, \
            open(os.path.join(out, name), "w", encoding="utf-8", newline="") as g:
        header = f.readline()
        if header.strip() != header_take.strip():
            raise SystemExit(f"{name}: headers differ: {header.strip()!r} vs {header_take.strip()!r}")
        g.write(header)
        seen = set()
        for line in f:
            s1 = line.split("\t", 1)[0]
            seen.add(s1)
            c = ctry.get(s1, "?")
            if c in keep:
                new = rows.get(s1)
                if new is None:
                    stats["missing_in_take"] += 1
                    new = line
                else:
                    stats["changed" if new.rstrip("\r\n") != line.rstrip("\r\n") else "unchanged"] += 1
                g.write(new if new.endswith("\n") else new + "\n")
                stats[f"from take ({c})"] += 1
            else:
                g.write(line if line.endswith("\n") else line + "\n")
                stats[f"from base ({c})"] += 1
    stats["take rows not in base"] = len(set(rows) - seen)
    print(f"{name}: " + ", ".join(f"{k} {v:,}" for k, v in sorted(stats.items())))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--test-s1", required=True)
    ap.add_argument("--base", required=True)
    ap.add_argument("--take", required=True)
    ap.add_argument("--countries", required=True, help="comma-separated, e.g. France")
    ap.add_argument("--out", required=True)
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    ctry = s1_countries(a.test_s1)
    keep = {c.strip() for c in a.countries.split(",")}
    unknown = keep - set(ctry.values())
    if unknown:
        raise SystemExit(f"unknown countries {unknown}; test has {sorted(set(ctry.values()))}")
    for name in ("matching_results.tsv", "candidate_pairs.tsv"):
        combine(name, a.base, a.take, a.out, ctry, keep)
    print(f"wrote {a.out}/matching_results.tsv and candidate_pairs.tsv - run utils/validate_submission.py on them")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile france_filter.py
"""Post-processing for the unseen country: drop matched pairs whose pool name ADDS a decoy marker word.

Train evidence (dev candidates, close pairs): a name that gains the country name ('india'), 'group' or 'holdings'
is a true copy 0.0% of the time (9.5k, 19.9k, 16.6k pairs), with or without a dropped word. The French test
equivalents are 'france', 'groupe', 'holding(s)', 'participations', 'developpement'; the v3 stem fallback only
partly catches them (v3.1 still accepts ~10k French pairs that add one). Generic suffixes that true copies do
add ('services', 'associes', 'fils', 'cie', legal forms) are NOT touched.

  python scripts/france_filter.py --matching <in matching_results.tsv> --test-dir <data>/test --out <out.tsv>
Plain Python; streams the files."""
import argparse
import re
import unicodedata
from collections import Counter

WORDS = {"france", "groupe", "holding", "holdings", "participations", "participation", "developpement"}
TOK = re.compile(r"[a-z0-9]+")


def toks(s):
    return set(TOK.findall(unicodedata.normalize("NFKD", s or "").encode("ascii", "ignore").decode().lower()))


def names(path, keep_country):
    out = {}
    with open(path, encoding="utf-8", newline="") as f:
        head = f.readline().rstrip("\r\n").split("\t")
        i_id, i_n, i_c = head.index("entity_id"), head.index("business_name"), head.index("country")
        for line in f:
            p = line.rstrip("\r\n").split("\t")
            if len(p) > max(i_id, i_n, i_c) and p[i_c] in keep_country:
                out[p[i_id]] = p[i_n]
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--matching", required=True)
    ap.add_argument("--test-dir", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--countries", default="France")
    ap.add_argument("--words", default=",".join(sorted(WORDS)))
    a = ap.parse_args()
    keep = {c.strip() for c in a.countries.split(",")}
    words = {w.strip() for w in a.words.split(",") if w.strip()}
    s1 = names(f"{a.test_dir}/test_source1.tsv", keep)
    pool = names(f"{a.test_dir}/test_source2.tsv", keep)
    pool.update(names(f"{a.test_dir}/test_source3.tsv", keep))
    s1_toks = {k: toks(v) for k, v in s1.items()}
    removed, kept, rows_changed = Counter(), 0, 0
    with open(a.matching, encoding="utf-8", newline="") as f, open(a.out, "w", encoding="utf-8", newline="") as g:
        g.write(f.readline())
        for line in f:
            sid, _, m = line.rstrip("\r\n").partition("\t")
            if sid not in s1_toks or not m:
                g.write(line if line.endswith("\n") else line + "\n")
                continue
            base = s1_toks[sid]
            out = []
            for p in m.split(","):
                hit = (toks(pool.get(p, "")) - base) & words
                if hit:
                    removed.update(hit)
                else:
                    out.append(p)
                    kept += 1
            rows_changed += len(out) != len(m.split(","))
            g.write(f"{sid}\t{','.join(out)}\n")
    print(f"{a.countries}: kept {kept:,} pairs, removed {sum(removed.values()):,} marker hits in {rows_changed:,} S1 rows: {dict(removed)}")


if __name__ == "__main__":
    main()

In [ ]:
import glob, sys, subprocess
I = "/kaggle/input"
t1 = glob.glob(f"{I}/**/test/test_source1.tsv", recursive=True); assert t1, "attach the challenge data"
DATA = os.path.dirname(os.path.dirname(t1[0]))
runs = [os.path.dirname(os.path.dirname(p)) for p in glob.glob(f"{I}/**/run.log", recursive=True)]
fr = [r for r in runs if "[france]" in open(f"{r}/run.log", errors="ignore").read()]
assert len(fr) == 1, f"attach exactly one France-bundle job output; found {fr}"
FR = os.path.join(fr[0], "output")
base = sorted({os.path.dirname(p) for p in glob.glob(f"{I}/**/matching_results.tsv", recursive=True)} - {FR})
pref = [b for b in base if b.endswith("submission_base_fr")] or [b for b in base if b.endswith("submission_base")] or base
BASE = pref[0]
print("base:", BASE, "\nfrance:", FR)
os.makedirs("/kaggle/working/tmp", exist_ok=True)
print(subprocess.run([sys.executable, "combine_by_country.py", "--test-s1", t1[0], "--base", BASE, "--take", FR,
                      "--countries", "France", "--out", "/kaggle/working/tmp"], capture_output=True, text=True).stdout)
os.makedirs("/kaggle/working/submission_france", exist_ok=True)
print(subprocess.run([sys.executable, "france_filter.py", "--matching", "/kaggle/working/tmp/matching_results.tsv",
                      "--test-dir", f"{DATA}/test", "--out", "/kaggle/working/submission_france/matching_results.tsv"],
                     capture_output=True, text=True).stdout)
os.replace("/kaggle/working/tmp/candidate_pairs.tsv", "/kaggle/working/submission_france/candidate_pairs.tsv")
V = (glob.glob(os.path.dirname(DATA) + "/utils/validate_submission.py") or glob.glob(f"{I}/**/validate_submission.py", recursive=True) or [None])[0]
if V:
    r = subprocess.run([sys.executable, V, "--matching", "/kaggle/working/submission_france/matching_results.tsv",
                        "--candidate", "/kaggle/working/submission_france/candidate_pairs.tsv", "--test-dir", f"{DATA}/test"], capture_output=True, text=True)
    print(r.stdout.strip().splitlines()[-1] if r.stdout.strip() else r.stderr[-500:])